In [2]:
import io

import pandas as pd
import requests


def poll_horizons_for_orion_spacecraft():
    """
    Polls the NASA JPL Horizons API specifically for the Artemis II Orion 
    spacecraft (ID: -1024). Returns the ephemeris data as a Pandas DataFrame.
    """
    url = 'https://ssd.jpl.nasa.gov/api/horizons.api'
    
    # Define parameters for the API call
    params = {
        'format': 'json',
        'COMMAND': "'-1024'",        # Exact ID for the Artemis II Orion Spacecraft
        'OBJ_DATA': 'YES',           # Return object summary
        'MAKE_EPHEM': 'YES',         # Generate ephemeris
        #'EPHEM_TYPE': 'OBSERVER',    # Observer-based ephemeris
        'EPHEM_TYPE': 'VECTORS',     # Change from 'OBSERVER'
        'OUT_UNITS': "'KM-S'",       # Force output to Kilometers and Seconds
        'CENTER': "'500@399'",       # Geocentric observer (Earth)
        'START_TIME': "'2026-04-02 02:00'", # Just after the 01:57 start time
        'STOP_TIME': "'2026-04-10'",        # Or whichever day you want the data to end
        'STEP_SIZE': "'1 h'",        # Step size (1 hour is better for spacecraft)
        #'QUANTITIES': "'1,20'",      # 1=Astrometric RA/DEC, 20=Observer range & range-rate
        'CSV_FORMAT': 'YES'          # Makes parsing into Pandas much easier
    }
    
    print("Polling JPL Horizons API for the Orion Spacecraft (-1024)...")
    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        print(f"HTTP Error: {response.status_code}")
        return None
        
    data = response.json()
    
    if 'error' in data:
        print("\nAPI Error:")
        print(data['error'])
        return None
        
    result_text = data.get('result', '')
    
    try:
        # Locate the tabular data wrapped between $$SOE and $$EOE markers
        start_idx = result_text.find('$$SOE')
        end_idx = result_text.find('$$EOE')
        
        if start_idx == -1 or end_idx == -1:
            print("Could not find ephemeris data markers ($$SOE / $$EOE).")
            print("This usually means the spacecraft is out of range for the dates provided.")
            return None
            
        # Extract the CSV block (+6 to skip the '$$SOE\n' string)
        csv_data = result_text[start_idx + 6 : end_idx]
        
        # Read the block into a Pandas DataFrame
        df = pd.read_csv(io.StringIO(csv_data), skipinitialspace=True)
        
        # Horizons adds a trailing comma to rows, creating an empty 'Unnamed' column.
        df = df.dropna(axis=1, how='all')
        
        print("Success! Data loaded into DataFrame.")
        return df
        
    except Exception as e:
        print(f"Error parsing data: {e}")
        return None

# --- Execute the function ---
df_orion = poll_horizons_for_orion_spacecraft()

# Display the dataframe nicely in Jupyter
if df_orion is not None:
    display(df_orion.head())

Polling JPL Horizons API for the Orion Spacecraft (-1024)...
Success! Data loaded into DataFrame.


,2461132.583333333,A.D. 2026-Apr-02 02:00:00.0000,-2.460259791031141E+04,-1.466164717271550E+04,-1.313590155061838E+03,-1.807002964285140E+00,-3.851912040453985E+00,-3.344134326180390E-01,9.563331130446044E-02,2.867014546264339E+04,3.535790390579346E+00
0,2.461133e+06,A.D. 2026-Apr-02 03:00:00.0000,-29092.645925,-27111.217203,-2391.890992,-0.820545,-3.110829,-0.268743,0.132888,39838.690838,2.732344
1,2.461133e+06,A.D. 2026-Apr-02 04:00:00.0000,-31024.033500,-37309.243469,-3274.577787,-0.300057,-2.578978,-0.222838,0.162223,48633.251614,2.184892
2,2.461133e+06,A.D. 2026-Apr-02 05:00:00.0000,-31496.027069,-45813.874956,-4008.424334,0.017037,-2.160043,-0.186121,0.185930,55740.275611,1.779133
3,2.461133e+06,A.D. 2026-Apr-02 06:00:00.0000,-31012.549198,-52942.404170,-4621.830702,0.239978,-1.809130,-0.155448,0.205245,61530.786490,1.447338
4,2.461133e+06,A.D. 2026-Apr-02 07:00:00.0000,-29833.830280,-58891.428767,-5132.244771,0.407634,-1.501706,-0.128631,0.220874,66216.295183,1.161896
